# Bài 2: Pandas — Series, DataFrame, lọc & biến đổi dữ liệu

Nếu NumPy là "mảng số", thì **Pandas là bảng dữ liệu** — giống Excel/SQL nhưng lập trình được. Trong thực tế, hầu hết dữ liệu AI bắt đầu từ file CSV/Excel/database: bạn cần đọc, khám phá, làm sạch và biến đổi nó **trước khi** đưa vào mô hình. Pandas là công cụ cho toàn bộ công đoạn này.

Mục tiêu bài học:
- `Series` (cột) và `DataFrame` (bảng)
- Khám phá nhanh dữ liệu: `head`, `info`, `describe`, `shape`
- Chọn cột, chọn hàng với `loc` / `iloc`
- Lọc hàng bằng điều kiện (boolean mask)
- Tạo/biến đổi cột, `apply`, `map`
- Sắp xếp, đếm giá trị, đọc/ghi CSV

> Nhắc lại cách học: đọc lý thuyết → chạy thử ví dụ → tự làm bài tập, đừng xem lời giải trước.

## 1. Series — một cột dữ liệu có nhãn

`Series` giống array 1 chiều của NumPy nhưng có thêm **index** (nhãn cho từng phần tử). Mọi phép toán vectorized của NumPy (bài 1) đều dùng được.

In [1]:
import numpy as np
import pandas as pd

s = pd.Series([0.45, 0.91, 0.72], index=["mẫu_a", "mẫu_b", "mẫu_c"], name="score")
print(s)
print()
print(s["mẫu_b"])         # truy cập theo nhãn
print(s.values)           # array NumPy bên dưới
print(s * 100)            # vectorized như NumPy
print(s.mean(), s.max())

mẫu_a    0.45
mẫu_b    0.91
mẫu_c    0.72
Name: score, dtype: float64

0.91
[0.45 0.91 0.72]
mẫu_a    45.0
mẫu_b    91.0
mẫu_c    72.0
Name: score, dtype: float64
0.6933333333333334 0.91


## 2. DataFrame — bảng dữ liệu

`DataFrame` là tập các `Series` (cột) dùng chung một index (hàng). Cách tạo phổ biến nhất: từ **dict của list** (mỗi key là một cột) — nhớ lại dict ở bài 5.

In [2]:
df = pd.DataFrame({
    "name":   ["An", "Bình", "Chi", "Dũng", "Em", "Giang"],
    "age":    [22, 25, 31, 28, 24, 35],
    "city":   ["Hà Nội", "TP.HCM", "Hà Nội", "Đà Nẵng", "TP.HCM", "Hà Nội"],
    "salary": [12.5, 18.0, 30.0, 22.5, 15.0, 40.0],   # triệu VND
    "is_ai":  [True, False, True, True, False, True],
})
df

,name,age,city,salary,is_ai
0,An,22,Hà Nội,12.5,True
1,Bình,25,TP.HCM,18.0,False
2,Chi,31,Hà Nội,30.0,True
3,Dũng,28,Đà Nẵng,22.5,True
4,Em,24,TP.HCM,15.0,False
5,Giang,35,Hà Nội,40.0,True


### Khám phá nhanh — việc đầu tiên khi nhận dữ liệu mới

In [3]:
print("shape:", df.shape)          # (số hàng, số cột)
print("cột :", list(df.columns))
print(df.dtypes)
print()
print(df.head(3))                  # 3 hàng đầu (tail(n) cho n hàng cuối)
print()
df.info()                          # tổng quan: kiểu dữ liệu, số giá trị không-null
print()
print(df.describe())               # thống kê các cột số: count, mean, std, min, quartile, max

shape: (6, 5)
cột : ['name', 'age', 'city', 'salary', 'is_ai']
name          str
age         int64
city          str
salary    float64
is_ai        bool
dtype: object

   name  age    city  salary  is_ai
0    An   22  Hà Nội    12.5   True
1  Bình   25  TP.HCM    18.0  False
2   Chi   31  Hà Nội    30.0   True

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   name    6 non-null      str    
 1   age     6 non-null      int64  
 2   city    6 non-null      str    
 3   salary  6 non-null      float64
 4   is_ai   6 non-null      bool   
dtypes: bool(1), float64(1), int64(1), str(2)
memory usage: 402.0 bytes

            age     salary
count   6.00000   6.000000
mean   27.50000  23.000000
std     4.84768  10.368221
min    22.00000  12.500000
25%    24.25000  15.750000
50%    26.50000  20.250000
75%    30.25000  28.125000
max    35.00000  40.000000


## 3. Chọn cột

- `df["col"]` → một `Series`
- `df[["col1", "col2"]]` → một `DataFrame` con (ngoặc **kép**: truyền vào một list tên cột)

In [4]:
print(df["name"].tolist())

sub = df[["name", "salary"]]
print(sub)
print(type(df["age"]).__name__, type(sub).__name__)

['An', 'Bình', 'Chi', 'Dũng', 'Em', 'Giang']
    name  salary
0     An    12.5
1   Bình    18.0
2    Chi    30.0
3   Dũng    22.5
4     Em    15.0
5  Giang    40.0
Series DataFrame


## 4. Chọn hàng: `loc` và `iloc`

Đây là chỗ người mới hay nhầm nhất:

| | Dựa vào | Slice cuối | Ví dụ |
|---|---|---|---|
| `loc`  | **nhãn** (label) của index/cột | **gồm** điểm cuối | `df.loc[1:3, "name"]` |
| `iloc` | **vị trí** (số thứ tự, từ 0) | **không gồm** điểm cuối (như list) | `df.iloc[1:3, 0]` |

Cú pháp chung: `df.loc[hàng, cột]` / `df.iloc[hàng, cột]`.

In [5]:
print(df.iloc[0])                     # hàng đầu tiên (ra một Series)
print()
print(df.iloc[1:3, 0:2])              # hàng 1-2, cột 0-1 (không gồm 3 và 2)
print()
print(df.loc[1:3, "name"])            # nhãn 1,2,3 -- GỒM cả 3
print()
print(df.loc[0, ["name", "city"]])    # một hàng, vài cột theo tên

name          An
age           22
city      Hà Nội
salary      12.5
is_ai       True
Name: 0, dtype: object

   name  age
1  Bình   25
2   Chi   31

1    Bình
2     Chi
3    Dũng
Name: name, dtype: str

name        An
city    Hà Nội
Name: 0, dtype: str


## 5. Lọc hàng theo điều kiện

Giống boolean mask của NumPy: điều kiện trên cột cho `Series` True/False, dùng làm index để giữ hàng `True`. Kết hợp bằng `&`, `|`, `~` và **bắt buộc có ngoặc** quanh từng điều kiện.

Ngoài ra: `isin([...])` kiểm tra thuộc tập, `str.contains(...)` cho chuỗi, và `query("...")` viết điều kiện như một chuỗi cho gọn.

In [6]:
print(df[df["age"] > 25])
print()

# Nhiều điều kiện: ở Hà Nội VÀ lương >= 30
print(df[(df["city"] == "Hà Nội") & (df["salary"] >= 30)])
print()

# isin: thuộc một tập giá trị
print(df[df["city"].isin(["TP.HCM", "Đà Nẵng"])][["name", "city"]])
print()

# is_ai là cột bool nên dùng thẳng; ~ là phủ định
print(df[~df["is_ai"]]["name"].tolist())
print()

# query: cùng kết quả, dễ đọc hơn khi điều kiện dài
print(df.query("age < 30 and is_ai"))

    name  age     city  salary  is_ai
2    Chi   31   Hà Nội    30.0   True
3   Dũng   28  Đà Nẵng    22.5   True
5  Giang   35   Hà Nội    40.0   True

    name  age    city  salary  is_ai
2    Chi   31  Hà Nội    30.0   True
5  Giang   35  Hà Nội    40.0   True

   name     city
1  Bình   TP.HCM
3  Dũng  Đà Nẵng
4    Em   TP.HCM

['Bình', 'Em']

   name  age     city  salary  is_ai
0    An   22   Hà Nội    12.5   True
3  Dũng   28  Đà Nẵng    22.5   True


## 6. Tạo và biến đổi cột

- **Vectorized** (ưu tiên, nhanh): `df["new"] = df["a"] * 2`
- **`map`**: ánh xạ từng giá trị của **một cột** qua dict hoặc hàm
- **`apply`**: áp hàm lên từng giá trị của cột (hoặc từng hàng với `axis=1`) — linh hoạt nhưng chậm hơn, chỉ dùng khi không có cách vectorized

Thao tác trên bản sao `df2 = df.copy()` để giữ nguyên `df` gốc cho các phần sau.

In [7]:
df2 = df.copy()

# Vectorized: đổi triệu VND -> VND, cột mới
df2["salary_vnd"] = df2["salary"] * 1_000_000

# Điều kiện dạng if/else bằng np.where (bài 1)
df2["level"] = np.where(df2["salary"] >= 25, "senior", "junior")

# map với dict: mã hoá nhãn chữ thành số (rất hay dùng trước khi train)
city_to_idx = {"Hà Nội": 0, "TP.HCM": 1, "Đà Nẵng": 2}
df2["city_idx"] = df2["city"].map(city_to_idx)

# apply với hàm: xử lý từng giá trị
df2["name_len"] = df2["name"].apply(len)

# Xử lý chuỗi cả cột bằng .str (không cần vòng for)
df2["name_upper"] = df2["name"].str.upper()

print(df2[["name", "salary_vnd", "level", "city_idx", "name_len", "name_upper"]])

# Xoá cột
df2 = df2.drop(columns=["salary_vnd", "name_upper"])
print(list(df2.columns))

    name  salary_vnd   level  city_idx  name_len name_upper
0     An  12500000.0  junior         0         2         AN
1   Bình  18000000.0  junior         1         4       BÌNH
2    Chi  30000000.0  senior         0         3        CHI
3   Dũng  22500000.0  junior         2         4       DŨNG
4     Em  15000000.0  junior         1         2         EM
5  Giang  40000000.0  senior         0         5      GIANG
['name', 'age', 'city', 'salary', 'is_ai', 'level', 'city_idx', 'name_len']


## 7. Sắp xếp, đếm giá trị, giá trị duy nhất

In [8]:
# Sắp xếp theo lương giảm dần
print(df.sort_values("salary", ascending=False)[["name", "salary"]])
print()

# Sắp xếp theo nhiều cột: city tăng dần, trong cùng city thì age giảm dần
print(df.sort_values(["city", "age"], ascending=[True, False])[["city", "name", "age"]])
print()

# Đếm số lần xuất hiện -- cực hữu ích để xem phân bố nhãn / mất cân bằng lớp
print(df["city"].value_counts())
print()
print(df["city"].unique())
print("số thành phố khác nhau:", df["city"].nunique())

# Thống kê trực tiếp trên cột / trên mask
print("lương trung bình:", df["salary"].mean())
print("tỉ lệ làm AI:", df["is_ai"].mean())      # True = 1 nên mean là tỉ lệ

    name  salary
5  Giang    40.0
2    Chi    30.0
3   Dũng    22.5
1   Bình    18.0
4     Em    15.0
0     An    12.5

      city   name  age
5   Hà Nội  Giang   35
2   Hà Nội    Chi   31
0   Hà Nội     An   22
1   TP.HCM   Bình   25
4   TP.HCM     Em   24
3  Đà Nẵng   Dũng   28

city
Hà Nội     3
TP.HCM     2
Đà Nẵng    1
Name: count, dtype: int64

<ArrowStringArray>
['Hà Nội', 'TP.HCM', 'Đà Nẵng']
Length: 3, dtype: str
số thành phố khác nhau: 3
lương trung bình: 23.0
tỉ lệ làm AI: 0.6666666666666666


## 8. Đọc và ghi CSV

`pd.read_csv("duong_dan.csv")` đọc file thành DataFrame; `df.to_csv("ra.csv", index=False)` ghi ra (`index=False` để không ghi cột index thừa). Để ví dụ chạy được mà không cần file có sẵn, ta đọc CSV từ một **chuỗi** bằng `io.StringIO` — cách dùng y hệt khi truyền đường dẫn file.

In [9]:
import io

csv_text = '''id,text,label
1,tuyệt vời,positive
2,tệ quá,negative
3,bình thường,neutral
4,rất hài lòng,positive
'''

reviews = pd.read_csv(io.StringIO(csv_text))
print(reviews)
print(reviews["label"].value_counts())

# Ghi ra file (bỏ comment để chạy thật):
# reviews.to_csv("reviews.csv", index=False)
# Đọc với tuỳ chọn hay dùng:
# pd.read_csv("reviews.csv", usecols=["text", "label"], nrows=100)

   id          text     label
0   1     tuyệt vời  positive
1   2        tệ quá  negative
2   3   bình thường   neutral
3   4  rất hài lòng  positive
label
positive    2
negative    1
neutral     1
Name: count, dtype: int64


## Bài tập thực hành

Dùng DataFrame sau cho cả các bài tập:

```python
students = pd.DataFrame({
    "name":  ["An", "Bình", "Chi", "Dũng", "Em", "Giang", "Hà", "Khôi"],
    "major": ["AI", "Data", "AI", "Web", "Data", "AI", "Web", "AI"],
    "gpa":   [3.5, 3.2, 3.8, 2.9, 3.6, 3.1, 3.4, 3.9],
    "year":  [3, 2, 4, 1, 3, 2, 4, 3],
})
```

1. In ra `shape`, tên các cột và 3 hàng cuối của `students`.
2. Chọn 2 cột `name` và `gpa`, in ra DataFrame con đó.
3. Lọc các sinh viên có `gpa >= 3.5` và thuộc ngành `"AI"`. In ra tên của họ.
4. Thêm cột `honor`: `True` nếu `gpa >= 3.6`, ngược lại `False` (không dùng vòng `for`). Có bao nhiêu sinh viên `honor`?
5. Dùng `map` với dict `{"AI": 0, "Data": 1, "Web": 2}` tạo cột `major_idx`.
6. Sắp xếp theo `gpa` giảm dần và in ra tên 3 sinh viên đứng đầu.
7. **(Nâng cao)** Dùng `value_counts` để đếm số sinh viên mỗi ngành, rồi tìm ngành có `gpa` trung bình cao nhất. Gợi ý: lọc từng ngành bằng mask rồi `.mean()`, hoặc thử `students.groupby("major")["gpa"].mean()` (sẽ học kỹ ở bài sau).

In [10]:
import pandas as pd

students = pd.DataFrame({
    "name":  ["An", "Bình", "Chi", "Dũng", "Em", "Giang", "Hà", "Khôi"],
    "major": ["AI", "Data", "AI", "Web", "Data", "AI", "Web", "AI"],
    "gpa":   [3.5, 3.2, 3.8, 2.9, 3.6, 3.1, 3.4, 3.9],
    "year":  [3, 2, 4, 1, 3, 2, 4, 3],
})

In [11]:
# Bài 1: shape, columns, tail(3)

In [12]:
# Bài 2: chọn cột name, gpa

In [13]:
# Bài 3: gpa >= 3.5 và major == 'AI'

In [14]:
# Bài 4: cột honor

In [15]:
# Bài 5: map major -> major_idx

In [16]:
# Bài 6: sort_values theo gpa, top 3

In [17]:
# Bài 7 (nâng cao): số SV mỗi ngành + ngành có gpa TB cao nhất

## Gợi ý

- Bài 1: `students.shape`, `students.columns`, `students.tail(3)`.
- Bài 3: `students[(students["gpa"] >= 3.5) & (students["major"] == "AI")]["name"]` — nhớ ngoặc quanh từng điều kiện.
- Bài 4: `students["honor"] = students["gpa"] >= 3.6`, rồi `students["honor"].sum()`.
- Bài 6: `students.sort_values("gpa", ascending=False).head(3)["name"]`.
- Bài 7: `students["major"].value_counts()` và `students.groupby("major")["gpa"].mean().idxmax()`.

## Lời giải mẫu

⚠️ Chỉ chạy ô dưới đây **sau khi** bạn đã tự làm xong 7 bài tập ở trên.

In [18]:
# --- Lời giải mẫu ---

# Bài 1
print(students.shape)
print(list(students.columns))
print(students.tail(3))

# Bài 2
print(students[["name", "gpa"]])

# Bài 3
top_ai = students[(students["gpa"] >= 3.5) & (students["major"] == "AI")]
print(top_ai["name"].tolist())

# Bài 4
students["honor"] = students["gpa"] >= 3.6
print("số SV honor:", students["honor"].sum())

# Bài 5
students["major_idx"] = students["major"].map({"AI": 0, "Data": 1, "Web": 2})
print(students[["major", "major_idx"]].drop_duplicates())

# Bài 6
print(students.sort_values("gpa", ascending=False).head(3)["name"].tolist())

# Bài 7
print(students["major"].value_counts())
avg_gpa = students.groupby("major")["gpa"].mean()
print(avg_gpa)
print("ngành gpa TB cao nhất:", avg_gpa.idxmax())

(8, 4)
['name', 'major', 'gpa', 'year']
    name major  gpa  year
5  Giang    AI  3.1     2
6     Hà   Web  3.4     4
7   Khôi    AI  3.9     3
    name  gpa
0     An  3.5
1   Bình  3.2
2    Chi  3.8
3   Dũng  2.9
4     Em  3.6
5  Giang  3.1
6     Hà  3.4
7   Khôi  3.9
['An', 'Chi', 'Khôi']
số SV honor: 3
  major  major_idx
0    AI          0
1  Data          1
3   Web          2
['Khôi', 'Chi', 'Em']
major
AI      4
Data    2
Web     2
Name: count, dtype: int64
major
AI      3.575
Data    3.400
Web     3.150
Name: gpa, dtype: float64
ngành gpa TB cao nhất: AI


## Tổng kết

Bạn đã học được:
- ✅ `Series` (cột có nhãn) và `DataFrame` (bảng); tạo từ dict của list
- ✅ Khám phá dữ liệu: `shape`, `dtypes`, `head`/`tail`, `info`, `describe`
- ✅ Chọn cột `df["a"]` / `df[["a", "b"]]`, chọn hàng bằng `loc` (nhãn, gồm điểm cuối) và `iloc` (vị trí)
- ✅ Lọc bằng boolean mask với `&`, `|`, `~`, `isin`, `query`
- ✅ Tạo/biến đổi cột bằng vectorized, `np.where`, `map`, `apply`, `.str`
- ✅ `sort_values`, `value_counts`, `unique`, đọc/ghi CSV

**Bài tiếp theo:** Bài 3 — Pandas nâng cao: `groupby`, `merge`/`join`, xử lý dữ liệu thiếu (missing data). Làm xong bài tập ở trên rồi báo mình nhé!